In [2]:
with open("Sherlock_Holmes.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 3868122




                          THE COMPLETE SHERLOCK HOLMES

                               Arthur C


In [3]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [5]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [12]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|-|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['THE', 'COMPLETE', 'SHERLOCK', 'HOLMES', 'Arthur', 'Conan', 'Doyle', 'Table', 'of', 'contents', 'A', 'Study', 'In', 'Scarlet', 'The', 'Sign', 'of', 'the', 'Four', 'The', 'Adventures', 'of', 'Sherlock', 'Holmes', 'A', 'Scandal', 'in', 'Bohemia', 'The', 'Red']


In [13]:
print(len(preprocessed))

807987


### Next we sort the vocabulary in alphabetical order and assign them token IDs

In [14]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words) 
print("Vocabulary size:", vocab_size)

Vocabulary size: 21194


In [15]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

('!', 0)
('"', 1)
('&', 2)
("'", 3)
('(', 4)
(')', 5)
('*1', 6)
(',', 7)
('-', 8)
('--', 9)
('.', 10)
('//sherlock', 11)
('000', 12)
('1', 13)
('10', 14)
('100', 15)
('1000', 16)
('104', 17)
('109', 18)
('10s', 19)
('10th', 20)


In [43]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|-|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations- Eg:"Hello , world !" -> "Hello, world!"
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [44]:
tokenizer = SimpleTokenizerV1(vocab)

text = """So alarming did the state of my finances become, that I soon realized that I must either leave the metropolis and rusticate somewhere in the country, or that I must make a complete alteration in my style of living."""
ids = tokenizer.encode(text)
print(ids)

[3587, 4694, 8505, 19247, 18406, 14336, 13956, 10098, 5521, 7, 19244, 2021, 18083, 16155, 19244, 2021, 13939, 9202, 12855, 19247, 13596, 4840, 16981, 18076, 11910, 19247, 7680, 7, 14440, 19244, 2021, 13939, 13289, 4328, 7211, 4756, 11910, 13956, 18695, 14336, 13065, 10]


In [45]:
tokenizer.decode(ids)

'So alarming did the state of my finances become, that I soon realized that I must either leave the metropolis and rusticate somewhere in the country, or that I must make a complete alteration in my style of living.'

In [46]:
text1 = """ "That's a strange thing," remarked my companion; "you are the second man to-day that has used that expression to me." """
ids1 = tokenizer.encode(text1)
print(ids1)

[1, 3849, 3, 16989, 4328, 18579, 19288, 7, 1, 16436, 13956, 7178, 191, 1, 21143, 5021, 19247, 17277, 13309, 19456, 8, 8081, 19244, 11289, 20269, 19244, 9751, 19456, 13464, 10, 1]


In [47]:
tokenizer.decode(ids1)

'" That\' s a strange thing," remarked my companion ;" you are the second man to - day that has used that expression to me."'

### Unknown or non-encoded words introduced:

In [48]:
text2 = "Hello, do you like tea?"
ids2 = tokenizer.encode(text2)
print(ids2)

KeyError: 'Hello'

In [49]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [50]:
len(vocab.items())

21196

In [51]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('£88', 21191)
('à', 21192)
('écarté', 21193)
('<|endoftext|>', 21194)
('<|unk|>', 21195)


In [52]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [53]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [54]:
tokenizer.encode(text)

[21195,
 7,
 8820,
 21143,
 12984,
 19118,
 192,
 21194,
 2045,
 19247,
 18821,
 21195,
 14336,
 19247,
 14667,
 10]

In [55]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit <|unk|> of the palace.'

"Hello" and "terraces" are unknown tokens/words in the vocabulary.

In [8]:
import re
text = "This costs $5.99, which is quite expensive!"
res = re.sub(r'(\$)(\d.\d+)', r'\2 USD', text)
print(res)
name = "Thomas, Mark Stephen"
res1 = re.sub(r'(\w+),\s+(.*)', r'\2 \1', name)
print(res1)

This costs 5.99 USD, which is quite expensive!
Mark Stephen Thomas
